# Calculate leverage in PIM portofolios for BNP

In [30]:
# libraries, libraries!
import time
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
import re # to extract dates
from tqdm import tqdm
from constants import pthCmp
from utilities import timediff

In [31]:
# list the fund holdings files

start_time = time.time()
start_time_lvrg = time.time()
pthL = pthCmp + r"\BNP Leverage DD\Annual Review Data"
files = [
    f.name
    for f in Path(pthL).iterdir()
    if f.is_file()
    and f.name.startswith("Portfolio Analytics Report")
    and f.name.endswith(".xlsx")
]

# for file in files:
#     print(file)
print(f"{len(files)} files")

print(timediff(start_time, time.time()))

12 files
0.1sec


In [32]:
# list the funds

pthF = pthCmp + r"\BNP Leverage DD\Prescient additional information.xlsx"
df2 = pd.read_excel(pthF, usecols = "N")
funds = df2.iloc[:,0].unique()
print(len(funds), funds)

21 ['QIFFGIF' 'PCGEARF' 'PGCBF' 'PGCEF' 'PGPCEM_C' 'IPIPF' 'NFMWAGG' 'PEQ'
 'PPSBAL_C' 'PABS' 'PIMBAL' 'PSIF' 'MOMTAAHI' 'MOMTAALI' 'MOMTAAMI'
 'SAAMCAU' 'SAAMINC' 'SAAMMOD' 'BPROV' 'UCTRFBAL' 'UNISABAL']


In [33]:
# column names for the summary dataframe
column_names = ['Date', 'Fund', 'NAV', 'Exposure (Gross)', 'Exposure (Commitment)', 'Leverage (Gross)', 'Leverage (Commitment)', 'MV - CE']

# define exclusions, being cash and cash equivalents and synthetic cash, per AIFMD
excl = ['CASH', 'MONEY MARKET', 'UNKNOWN', 'SYTH']

In [34]:
# ### TEST

# # input to the for loop
# file = "Portfolio Analytics Report 31012025.xlsx"

# # dataframe that month's holdings
# fl_pth = pthL + '\\' + file
# df = pd.read_excel(fl_pth)

# # normalise fund holding percentages
# df['fund_mv_total'] = df.groupby('Entity ID')['Sum of Market Value Income'].transform('sum')
# df['% of Total Market Value'] = df['Sum of Market Value Income'] / df['fund_mv_total'] * 100
# df.groupby('Entity ID')['% of Total Market Value'].sum() # check

# df['fund_ce_total'] = df.groupby('Entity ID')['Current Exposure'].transform('sum')
# df['Current Exposure %'] = df['Current Exposure'] / df['fund_ce_total'] * 100
# df.groupby('Entity ID')['Current Exposure %'].sum()

# # # extract the report date
# # pattern = r"\b\d{8}\b"
# # match = re.search(pattern, file)
# # dt = datetime.strptime(match.group(), "%d%m%Y").date()

# dt = df.iloc[1, 35].date()

# print(dt)

# ### TEST

In [35]:
start_time = time.time()
print(f"\nCalculating {len(files) * len(funds)} exposures \
for {len(funds)} funds over {len(files)} periods ...")

vals = []
for file in tqdm(files):
    # dataframe that month's holdings
    fl_pth = pthL + '\\' + file
    df = pd.read_excel(fl_pth)
    
    # normalise fund holding percentages
    df['fund_mv_total'] = df.groupby('Entity ID')['Sum of Market Value Income'].transform('sum')
    df['% of Total Market Value'] = df['Sum of Market Value Income'] / df['fund_mv_total'] * 100
    # df.groupby('Entity ID')['% of Total Market Value'].sum() # check
    
    df['fund_ce_total'] = df.groupby('Entity ID')['Current Exposure'].transform('sum')
    df['Current Exposure %'] = df['Current Exposure'] / df['fund_ce_total'] * 100
    # df.groupby('Entity ID')['Current Exposure %'].sum()
    
    # # extract the report date from file name
    # match = re.search(pattern, file)
    # dt = datetime.strptime(match.group(), "%d%m%Y").date()
    
    dt = df.iloc[1, 35].date()
    
    # print(f"{fl_pth}", dt)

    # loop through each fund for that date
    for fund in funds:
        #dataframe the fund holdings
        df_x = df[df['Entity ID'] == fund]

        # calculate market value and effective exposure
        mv = df_x.loc[df_x["Entity ID"] == fund, "Sum of Market Value Income"].sum()
        ce = df_x.loc[df_x["Entity ID"] == fund, "Current Exposure"          ].sum()

        # sum exposure
        gross      = df_x.loc[(df_x["Entity ID"] == fund) & (~df_x["Valuation First Level"].isin(excl)), "Current Exposure"].abs().sum()
        commitment = abs(df_x.loc[(df_x["Entity ID"] == fund) & (~df_x["Valuation First Level"].isin(excl)), "Current Exposure"].sum())

        # calculate leverage
        leverage_gross      = gross / mv
        leverage_commitment = commitment / mv
     
        # populate the summary dataframe
        new_data = [dt, fund, mv, gross, commitment, leverage_gross, leverage_commitment, mv - ce]
        vals.append(new_data)

# dataframe the calculation results
summary_df = pd.DataFrame(vals)
summary_df.columns = column_names
summary_df = summary_df.sort_values(by = 'Date', ascending = False)
summary_df = summary_df.reset_index(drop = True)

print(f" {timediff(start_time, time.time())} calculating \
{len(files) * len(funds)} exposures for \
{len(funds)} funds over {len(files)} periods ...\n")


Calculating 252 exposures for 21 funds over 12 periods ...


100%|██████████████████████████████████████████████████████████████████████████████████| 12/12 [00:16<00:00,  1.38s/it]

 16.6sec calculating 252 exposures for 21 funds over 12 periods ...



In [38]:
summary_df

,Date,Fund,NAV,Exposure (Gross),Exposure (Commitment),Leverage (Gross),Leverage (Commitment),MV - CE
0,2025-12-31,UNISABAL,1.665250e+09,1.646816e+09,1.646816e+09,0.988930,0.988930,0.000000e+00
1,2025-12-31,PIMBAL,8.631153e+09,1.344291e+10,8.947074e+09,1.557487,1.036602,0.000000e+00
2,2025-12-31,QIFFGIF,1.297970e+08,3.522057e+08,1.094057e+08,2.713513,0.842899,0.000000e+00
3,2025-12-31,PGCBF,2.004604e+08,3.371061e+08,1.686984e+08,1.681660,0.841555,0.000000e+00
4,2025-12-31,PGCEF,2.492626e+08,3.833681e+08,2.847081e+08,1.538009,1.142201,-5.960464e-08
...,...,...,...,...,...,...,...,...
247,2025-01-31,IPIPF,2.237854e+08,1.760729e+08,1.760729e+08,0.786794,0.786794,0.000000e+00
248,2025-01-31,PGPCEM_C,4.551388e+07,1.511632e+08,7.087738e+07,3.321256,1.557270,-7.450581e-09
249,2025-01-31,PGCEF,1.866230e+08,3.079688e+08,1.995123e+08,1.650218,1.069066,0.000000e+00
250,2025-01-31,PGCBF,1.558729e+08,3.109560e+08,1.563971e+08,1.994933,1.003363,2.980232e-08


In [37]:
fund = 'PIMBAL'
df_a = summary_df[summary_df['Fund'] == fund]
df_a

,Date,Fund,NAV,Exposure (Gross),Exposure (Commitment),Leverage (Gross),Leverage (Commitment),MV - CE
1,2025-12-31,PIMBAL,8.631153e+09,1.344291e+10,8.947074e+09,1.557487,1.036602,0.000000e+00
41,2025-11-30,PIMBAL,8.402768e+09,1.383106e+10,8.713273e+09,1.646012,1.036953,-9.536743e-07
53,2025-10-31,PIMBAL,8.234491e+09,1.181624e+10,8.569723e+09,1.434969,1.040711,0.000000e+00
75,2025-09-30,PIMBAL,7.961410e+09,9.541840e+09,8.307975e+09,1.198511,1.043531,0.000000e+00
93,2025-08-31,PIMBAL,7.649123e+09,9.219496e+09,7.941564e+09,1.205301,1.038232,0.000000e+00
115,2025-07-31,PIMBAL,7.402127e+09,8.544273e+09,7.570126e+09,1.154300,1.022696,0.000000e+00
136,2025-06-30,PIMBAL,7.214616e+09,8.401485e+09,7.433364e+09,1.164509,1.030320,-2.861023e-06
158,2025-05-31,PIMBAL,7.050828e+09,8.264889e+09,7.314585e+09,1.172187,1.037408,9.536743e-07
168,2025-04-30,PIMBAL,6.909693e+09,8.088724e+09,7.171954e+09,1.170634,1.037956,0.000000e+00
201,2025-03-31,PIMBAL,6.697636e+09,7.857776e+09,6.950825e+09,1.173216,1.037803,-9.536743e-07


In [26]:
# calculate averages and standard deviations over the entire period

start_time = time.time()
print(f"\nCalculating averages and standard deviations for the {len(funds)} funds")

# scale = np.sqrt(len(files))
scale = 1
rptDate = summary_df['Date'].max()
averages = []

for fund in tqdm(funds):
    df_a = summary_df[summary_df['Fund'] == fund]
    average_gross = df_a['Leverage (Gross)'].mean()
    average_commitment = df_a['Leverage (Commitment)'].mean()
    stddev_gross = df_a['Leverage (Gross)'].std() * scale
    stddev_commitment = df_a['Leverage (Commitment)'].std() * scale
    new_data = [rptDate, fund, average_gross, stddev_gross, average_commitment, stddev_commitment]
    averages.append(new_data)

# dataframe the calculation results
col_names = [f'{len(files)} months ended {rptDate.strftime("%d%b%Y")}', 
             'Fund', 'Average Leverage (Gross)', 'Std Dev Leverage (Gross)', 
            'Average Leverage (Commitment)', 'Std Dev Leverage (Commitment)']
leverages = pd.DataFrame(averages)
leverages.columns = col_names

# leverages

print(f"{timediff(start_time, time.time())} calculating averages and standard deviations for the {len(funds)} funds\n")


Calculating averages and standard deviations for the 21 funds


100%|████████████████████████████████████████████████████████████████████████████████| 21/21 [00:00<00:00, 1284.74it/s]

0.0sec calculating averages and standard deviations for the 21 funds



In [27]:
# write the summary and leverage dataframe to Excel
filename = pthL + '\\' + f'Fund Leverages {rptDate.strftime("%d%b%Y")}.xlsx'
with pd.ExcelWriter(filename, engine  = 'xlsxwriter') as writer:
    leverages.to_excel( writer, index = False,  sheet_name = 'averages' )
    summary_df.to_excel(writer, index = False,  sheet_name = 'exposures')
    df.to_excel(        writer, index = False,  sheet_name = 'holdings' )
    
writer.close()

print(filename)

print(f"\n\n{timediff(start_time_lvrg, time.time())} total time\n")

\\PIM-CPT-FS.prescient.local\PIM-Documents$\Investment Operations\GRC\Compliance\BNP Leverage DD\Annual Review Data\Fund Leverages 31Dec2025.xlsx


26.1sec total time



C:\Users\hilton.netta\AppData\Local\anaconda3\Lib\site-packages\xlsxwriter\workbook.py:368: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [28]:
# !jupyter nbconvert --to script leverage.ipynb # convert from .ipynb to .py